# 🔎 Prospector v3 — busca por raio

Escolha uma cidade e um raio em km. O sistema varre a área inteira, pega
também as cidades vizinhas, e devolve os comércios **sem site próprio**.

**Como usar:** clique no ▶️ de cada célula, de cima para baixo. A última baixa tudo num `.zip`.

| Célula | O que faz |
|---|---|
| 1 | Você preenche cidade, raio e ramo |
| 2 | Carrega o motor da busca |
| 3 | Roda a busca, filtra e baixa as fotos |
| 4 | Monta a planilha e o PDF |
| 5 | Gera um site por lead, com a cor tirada da fachada |
| 6 | Monta o painel de prospecção |
| 7 | Deixa você editar os sites conversando com o Claude |
| 8 | Baixa tudo |

---

### 🔑 Antes de começar: as duas chaves

As chaves **não ficam escritas no notebook**. Elas ficam no cofre do Colab,
então não vão junto quando você compartilha ou salva o arquivo.

No menu da esquerda, clique no 🔑 e adicione:

| Nome do secret | Para que serve | Onde pegar |
|---|---|---|
| `GOOGLE_PLACES_API_KEY` | buscar os comércios | console.cloud.google.com |
| `ANTHROPIC_API_KEY` | editar os sites com o Claude (célula 7) | console.anthropic.com |

Em cada uma, ligue a chavinha **"Acesso ao notebook"**.

> Só a primeira é obrigatória. Sem a segunda, tudo funciona menos a célula 7.

---

⚠️ **Sobre as fotos:** as imagens do perfil pertencem a quem as enviou ao Google.
Use para montar a proposta e mostrar o preview para o dono. Só publique em
site aberto depois do sim dele.

In [ ]:
#@title ⚙️ 1. Preencha aqui e clique no ▶️

# A chave NAO fica escrita aqui. Ela fica no cofre do Colab:
# menu da esquerda -> 🔑 -> "Adicionar novo secret"
#   Nome:  GOOGLE_PLACES_API_KEY
#   Valor: sua chave
#   E ligue a chavinha "Acesso ao notebook"
# Assim a chave nao vai junto quando voce compartilha ou salva o notebook.

CIDADE_CENTRO = "Sumaré" #@param {type:"string"}
ESTADO = "SP" #@param {type:"string"}
RAIO_KM = 20 #@param {type:"slider", min:5, max:100, step:5}
RAMOS = "pet shop" #@param {type:"string"}
NOTA_MINIMA = 4 #@param {type:"slider", min:3, max:5, step:0.1}
MIN_AVALIACOES = 20 #@param {type:"slider", min:0, max:200, step:5}
BAIXAR_FOTOS = True #@param {type:"boolean"}
MAX_FOTOS_POR_LOCAL = 8 #@param {type:"slider", min:1, max:10, step:1}
PAGINAS_POR_PONTO = 2 #@param {type:"slider", min:1, max:3, step:1}
LIMITE_CONSULTAS = 250 #@param {type:"slider", min:50, max:600, step:50}

try:
    from google.colab import userdata
    API_KEY = (userdata.get("GOOGLE_PLACES_API_KEY") or "").strip()
except Exception:
    API_KEY = ""

CONFIG = {
    "cidade": CIDADE_CENTRO.strip(),
    "estado": ESTADO.strip(),
    "raio_km": float(RAIO_KM),
    "sub_km": 20.0,
    "ramos": [r.strip() for r in RAMOS.split(",") if r.strip()],
    "nota_minima": float(NOTA_MINIMA),
    "min_avaliacoes": int(MIN_AVALIACOES),
    "baixar_fotos": BAIXAR_FOTOS,
    "max_fotos_por_local": int(MAX_FOTOS_POR_LOCAL),
    "paginas": int(PAGINAS_POR_PONTO),
    "limite": int(LIMITE_CONSULTAS),
    "largura_foto": 1600,
    "pasta_saida": "resultado",
    "idioma": "pt-BR",
}

print(f"Centro: {CONFIG['cidade']}/{CONFIG['estado']} — raio de {int(CONFIG['raio_km'])} km")
print(f"Ramos: {', '.join(CONFIG['ramos'])}")
print(f"Filtro: nota >= {CONFIG['nota_minima']} | avaliações >= {CONFIG['min_avaliacoes']} | sem site próprio")

if API_KEY:
    print(f"\nChave do Google: ok (termina em ...{API_KEY[-4:]})")
else:
    print("\n⚠️  Falta a chave do Google Places.")
    print("   Menu da esquerda → \U0001f511 → Adicionar novo secret")
    print("   Nome: GOOGLE_PLACES_API_KEY")
    print("   Depois ligue a chavinha 'Acesso ao notebook' e rode esta célula de novo.")

In [ ]:
#@title 🔧 2. Motor (só clique no ▶️)

import csv, json, math, re, time, unicodedata
from pathlib import Path
import requests

BASE = "https://places.googleapis.com/v1/places:searchText"

FIELDS = ",".join([
    "places.id", "places.displayName", "places.formattedAddress",
    "places.nationalPhoneNumber", "places.rating", "places.userRatingCount",
    "places.websiteUri", "places.googleMapsUri", "places.location",
    "places.primaryTypeDisplayName", "places.businessStatus", "places.photos",
    "nextPageToken",
])

SOCIAIS = (
    "instagram.com", "facebook.com", "fb.com", "linktr.ee", "linktree",
    "wa.me", "api.whatsapp.com", "whatsapp.com", "linkbio", "beacons.ai",
    "bio.link", "youtube.com", "tiktok.com", "ifood.com", "linkedin.com",
    "google.com", "sites.google.com", "business.site", "negocio.site",
)

def slug(t):
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
    return (re.sub(r"[^a-zA-Z0-9]+", "-", t).strip("-").lower())[:60] or "sem-nome"

def tem_site_proprio(url):
    if not url:
        return False
    u = url.lower()
    return not any(d in u for d in SOCIAIS)

def achar_instagram(p):
    url = (p.get("websiteUri") or "").lower()
    if "instagram.com" in url:
        h = url.rstrip("/").split("instagram.com/")[-1].split("?")[0]
        return "@" + h if h else url
    return ""

def distancia_km(la1, lo1, la2, lo2):
    R = 6371.0
    p1, p2 = math.radians(la1), math.radians(la2)
    dp, dl = p2 - p1, math.radians(lo2 - lo1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def geocodificar(cidade, estado):
    h = {"Content-Type": "application/json", "X-Goog-Api-Key": API_KEY,
         "X-Goog-FieldMask": "places.location,places.formattedAddress"}
    b = {"textQuery": f"{cidade}, {estado}, Brasil", "languageCode": "pt-BR", "pageSize": 1}
    r = requests.post(BASE, headers=h, json=b, timeout=30)
    if r.status_code != 200:
        print(f"Erro ao localizar a cidade: {r.status_code} \u2014 {r.text[:200]}")
        return None
    places = r.json().get("places", [])
    if not places:
        return None
    loc = places[0]["location"]
    return loc["latitude"], loc["longitude"], places[0].get("formattedAddress", "")

def gerar_pontos(lat0, lng0, raio_km, sub_km):
    passo = sub_km * 1.4
    n = int(math.ceil((raio_km + sub_km) / passo))
    pontos = []
    for i in range(-n, n + 1):
        for j in range(-n, n + 1):
            dy, dx = i * passo, j * passo
            if math.hypot(dx, dy) > raio_km + sub_km * 0.7:
                continue
            lat = lat0 + dy / 110.574
            lng = lng0 + dx / (111.320 * math.cos(math.radians(lat0)))
            pontos.append((lat, lng))
    return pontos

def buscar_ponto(ramo, lat, lng, cfg, contador):
    h = {"Content-Type": "application/json", "X-Goog-Api-Key": API_KEY,
         "X-Goog-FieldMask": FIELDS}
    # locationBias aceita c\u00edrculo; locationRestriction no Text Search s\u00f3 aceita ret\u00e2ngulo
    circulo = {"circle": {"center": {"latitude": lat, "longitude": lng},
                          "radius": cfg["sub_km"] * 1000}}
    out, token, pag = [], None, 0
    while pag < cfg["paginas"]:
        if contador["n"] >= cfg["limite"]:
            return out
        b = {"textQuery": ramo, "languageCode": cfg["idioma"], "pageSize": 20,
             "locationBias": circulo}
        if token:
            b["pageToken"] = token
        r = requests.post(BASE, headers=h, json=b, timeout=30)
        contador["n"] += 1
        if r.status_code != 200:
            print(f"    ! {r.status_code}: {r.text[:200]}")
            break
        d = r.json()
        out.extend(d.get("places", []))
        token = d.get("nextPageToken")
        pag += 1
        if not token:
            break
        time.sleep(2)
    return out

def baixar_fotos(place, destino, cfg):
    fotos = place.get("photos", [])[: cfg["max_fotos_por_local"]]
    if not fotos:
        return []
    destino.mkdir(parents=True, exist_ok=True)
    salvas = []
    for i, f in enumerate(fotos, 1):
        nome = f.get("name")
        if not nome:
            continue
        url = f"https://places.googleapis.com/v1/{nome}/media?maxWidthPx={cfg['largura_foto']}&key={API_KEY}"
        try:
            resp = requests.get(url, timeout=60)
            if resp.status_code != 200:
                continue
            arq = destino / f"{i:02d}.jpg"
            arq.write_bytes(resp.content)
            salvas.append(str(arq))
            autores = [a.get("displayName", "") for a in f.get("authorAttributions", [])]
            if autores:
                with open(destino / "creditos.txt", "a", encoding="utf-8") as fh:
                    fh.write(f"{arq.name}: {', '.join(autores)}\n")
        except requests.RequestException:
            continue
    return salvas

print("\u2705 Motor carregado.")

In [ ]:
#@title ▶️ 3. Rodar a busca

cfg = CONFIG
centro = geocodificar(cfg["cidade"], cfg["estado"])
if not centro:
    raise SystemExit("N\u00e3o consegui localizar a cidade. Confira o nome e a sigla do estado.")

lat0, lng0, endereco_centro = centro
pontos = gerar_pontos(lat0, lng0, cfg["raio_km"], cfg["sub_km"])
previsto = len(pontos) * len(cfg["ramos"]) * cfg["paginas"]

print(f"Centro: {endereco_centro}")
print(f"{len(pontos)} pontos de varredura para cobrir {int(cfg['raio_km'])} km")
print(f"At\u00e9 {previsto} consultas (teto de seguran\u00e7a: {cfg['limite']})\n")

base = Path(cfg["pasta_saida"]) / slug(f"{cfg['cidade']}-{cfg['estado']}-{int(cfg['raio_km'])}km")
base.mkdir(parents=True, exist_ok=True)

contador = {"n": 0}
vistos, brutos = set(), []
erros = 0

for ramo in cfg["ramos"]:
    print(f"> {ramo}")
    for idx, (la, lo) in enumerate(pontos, 1):
        if contador["n"] >= cfg["limite"]:
            print("  (teto de consultas atingido \u2014 parando)")
            break
        achados = buscar_ponto(ramo, la, lo, cfg, contador)
        if not achados:
            erros += 1
        novos = 0
        for p in achados:
            pid = p.get("id")
            if pid and pid not in vistos:
                vistos.add(pid)
                p["_ramo"] = ramo
                brutos.append(p)
                novos += 1
        print(f"  ponto {idx}/{len(pontos)} \u2014 +{novos} novos (total {len(brutos)})")
        if erros >= 3 and not brutos:
            print("\n  \u26a0\ufe0f V\u00e1rias buscas seguidas sem retorno \u2014 parando para n\u00e3o gastar \u00e0 toa.")
            break

print(f"\n{contador['n']} consultas feitas. Filtrando {len(brutos)} estabelecimentos...\n")

leads = []
for p in brutos:
    nota = p.get("rating") or 0
    n_aval = p.get("userRatingCount") or 0
    if nota < cfg["nota_minima"] or n_aval < cfg["min_avaliacoes"]:
        continue
    if tem_site_proprio(p.get("websiteUri")):
        continue
    if p.get("businessStatus") not in (None, "OPERATIONAL"):
        continue

    loc = p.get("location", {})
    dist = distancia_km(lat0, lng0, loc.get("latitude", lat0), loc.get("longitude", lng0))
    if dist > cfg["raio_km"]:
        continue

    nome = p.get("displayName", {}).get("text", "")
    pasta = base / "fotos" / f"{slug(nome)}-{p['id'][-6:]}"
    fotos = baixar_fotos(p, pasta, cfg) if cfg["baixar_fotos"] else []

    leads.append({
        "nome": nome,
        "telefone": p.get("nationalPhoneNumber", ""),
        "nota": nota,
        "avaliacoes": n_aval,
        "instagram": achar_instagram(p),
        "km_do_centro": round(dist, 1),
        "link_cadastrado": p.get("websiteUri", "") or "",
        "ramo_busca": p.get("_ramo", ""),
        "categoria_google": p.get("primaryTypeDisplayName", {}).get("text", ""),
        "endereco": p.get("formattedAddress", ""),
        "maps": p.get("googleMapsUri", ""),
        "qtd_fotos": len(fotos),
        "pasta_fotos": str(pasta) if fotos else "",
        "place_id": p.get("id", ""),
    })
    print(f"  \u2713 {nome} \u2014 {nota}\u2605 ({n_aval}) \u2014 {dist:.0f} km \u2014 {len(fotos)} fotos")

if leads:
    leads.sort(key=lambda x: (-x["avaliacoes"], -x["nota"]))
    with open(base / "leads.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(leads[0].keys()))
        w.writeheader()
        w.writerows(leads)
    (base / "leads.json").write_text(json.dumps(leads, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"\n{'-'*45}")
    print(f"{len(leads)} leads sem site em {int(cfg['raio_km'])} km de {cfg['cidade']}")
    print(f"{sum(1 for l in leads if l['telefone'])} com telefone | {sum(1 for l in leads if l['instagram'])} com Instagram")
    try:
        import pandas as pd
        display(pd.DataFrame(leads)[["nome", "telefone", "nota", "avaliacoes", "km_do_centro", "instagram", "qtd_fotos"]])
    except Exception:
        pass
elif brutos:
    print("\nAchei estabelecimentos, mas nenhum passou nos filtros.")
    print("Baixe a nota m\u00ednima ou o m\u00ednimo de avalia\u00e7\u00f5es e rode de novo.")
else:
    print("\nA busca n\u00e3o retornou nada. Veja as mensagens de erro acima.")

In [ ]:
#@title 📄 Planilha e PDF

!pip install -q fpdf2
from fpdf import FPDF
import csv

def limpo(s):
    return str(s).encode("latin-1", "ignore").decode("latin-1")

# ---------- CSV que abre certo no Excel brasileiro ----------
with open(base / "leads_excel.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["Nome", "Telefone", "Endereco", "Nota", "Avaliacoes", "Instagram", "Km"])
    for l in leads:
        w.writerow([l["nome"], l["telefone"], l["endereco"],
                    str(l["nota"]).replace(".", ","), l["avaliacoes"],
                    l["instagram"], str(l["km_do_centro"]).replace(".", ",")])

# ---------- PDF em tabela ----------
MARGEM = 10
COLS = [
    ("N",         8),
    ("NOME",     58),
    ("TELEFONE", 34),
    ("ENDERECO", 118),
    ("NOTA",     14),
    ("AVAL.",    16),
    ("KM",       12),
]
LARGURA = sum(c[1] for c in COLS)   # 260 mm, cabe em A4 deitado

pdf = FPDF(orientation="L", format="A4")
pdf.set_auto_page_break(auto=False)
pdf.set_margins(MARGEM, MARGEM, MARGEM)

def corta(texto, larg):
    t = limpo(texto)
    if pdf.get_string_width(t) <= larg - 3:
        return t
    while len(t) > 3 and pdf.get_string_width(t + "..") > larg - 3:
        t = t[:-1]
    return t + ".."

def cabecalho():
    pdf.add_page()
    pdf.set_xy(MARGEM, MARGEM)
    pdf.set_font("helvetica", "B", 14)
    pdf.set_text_color(0, 0, 0)
    pdf.cell(LARGURA, 8, limpo(f"Leads sem site - {CONFIG['cidade']}/{CONFIG['estado']}"),
             new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("helvetica", "", 9)
    pdf.set_text_color(110, 110, 110)
    pdf.cell(LARGURA, 5, limpo(f"{len(leads)} estabelecimentos - raio de {int(CONFIG['raio_km'])} km"),
             new_x="LMARGIN", new_y="NEXT")
    pdf.ln(3)

    pdf.set_font("helvetica", "B", 8)
    pdf.set_fill_color(45, 45, 45)
    pdf.set_text_color(255, 255, 255)
    for titulo, larg in COLS:
        pdf.cell(larg, 7, limpo(titulo), align="L", fill=True)
    pdf.ln(7)
    pdf.set_text_color(0, 0, 0)

cabecalho()

for i, l in enumerate(leads, 1):
    if pdf.get_y() > 180:
        cabecalho()

    y = pdf.get_y()
    if i % 2 == 0:
        pdf.set_fill_color(246, 246, 246)
        pdf.rect(MARGEM, y, LARGURA, 7, style="F")

    pdf.set_xy(MARGEM, y)
    valores = [
        str(i),
        l["nome"],
        l["telefone"] or "-",
        l["endereco"],
        str(l["nota"]).replace(".", ","),
        str(l["avaliacoes"]),
        str(l["km_do_centro"]).replace(".", ","),
    ]
    for (titulo, larg), valor in zip(COLS, valores):
        pdf.set_font("helvetica", "B" if titulo == "NOME" else "", 8)
        pdf.cell(larg, 7, corta(valor, larg), align="L")
    pdf.ln(7)

    pdf.set_draw_color(225, 225, 225)
    pdf.line(MARGEM, pdf.get_y(), MARGEM + LARGURA, pdf.get_y())

caminho = str(base / "leads.pdf")
pdf.output(caminho)
print(f"PDF gerado com {len(leads)} leads.")
print("Sai no .zip da última célula: leads.pdf e leads_excel.csv")

In [ ]:
#@title 🎨 Gerador de sites — a cor sai da fachada, do logo ou da paleta do nicho

import json, shutil, re, unicodedata, colorsys
from pathlib import Path
from string import Template
from html import escape as esc
from PIL import Image
from google.colab import files

# ─────────── cor de reserva de cada nicho ───────────
# Usada quando a foto nao entrega cor propria (fachada branca, foto escura,
# so o interior da loja). O resto da paleta e sempre derivado dela.
NICHO = {
    "pet":     "#3E8E5A",
    "oficina": "#C24E12",
    "clinica": "#1E6FA8",
    "comida":  "#A32B33",
    "beleza":  "#7A3E86",
    "escola":  "#1F6E63",
    "padrao":  "#2B5F8C",
}

SERVICOS = {
    "pet": [("Banho e tosa", "Banho, tosa, unhas e limpeza de ouvido. Atende do chihuahua ao rottweiler."),
            ("Rações e acessórios", "As rações que o bicho daqui já come. Se o seu come outra, a gente traz."),
            ("Atendimento próximo", "A gente fica aqui do lado. Seu pet é cliente, não número.")],
    "oficina": [("Diagnóstico honesto", "A gente mostra o que precisa fazer. Ninguém sai daqui sem entender o orçamento."),
                ("Manutenção preventiva", "Revisão, óleo, filtro, freios. A gente avisa quando está na hora de mexer."),
                ("Prazo que se cumpre", "Se a gente disser que fica pronto segunda, segunda fica. Sem surpresa.")],
    "clinica": [("Atendimento humano", "Consulta de verdade. A gente escuta e examina. Não é correria."),
                ("Estrutura completa", "Consultório limpo e equipado. A gente esteriliza tudo direitinho."),
                ("Agendamento fácil", "Marca por telefone e é atendido no horário. Sem demora.")],
    "comida": [("Feito na hora", "Nada congelado. A gente faz na hora em que você chega."),
               ("Receita da casa", "É a mesma receita desde o começo. Ninguém copiou."),
               ("Salão e retirada", "Tem mesa para comer aqui ou você leva para casa.")],
    "beleza": [("Profissionais formados", "Todo mundo aqui fez curso. A mão é leve e experiente."),
               ("Produtos de linha", "A gente usa produto profissional, não o barato de supermercado."),
               ("Seu horário respeitado", "Você marca a hora e a gente está pronto. Nada de atraso.")],
    "escola": [("Turmas pequenas", "Poucos alunos por sala. O professor sabe o nome de cada um."),
               ("Equipe que fica", "Os professores ficam anos aqui. Conhecem as famílias."),
               ("Portas abertas", "Venha conhecer a escola antes de decidir. Sem hora marcada.")],
    "padrao": [("Atendimento próximo", "A gente é daqui. Conhece o bairro e conhece você."),
               ("Qualidade comprovada", "A gente sobrevive porque faz certo. Reputação não se compra."),
               ("Fácil de encontrar", "A gente está aqui do lado. Perto de casa, fácil de achar.")],
}

def familia(t):
    r = t.lower()
    if any(k in r for k in ("pet", "veterin", "animal", "agro")): return "pet"
    if any(k in r for k in ("oficina", "mecan", "auto", "funilar", "pneu")): return "oficina"
    if any(k in r for k in ("clinic", "odonto", "dent", "saude", "fisio", "medic")): return "clinica"
    if any(k in r for k in ("restaur", "pizz", "lanch", "bar", "cafe", "padar", "espet")): return "comida"
    if any(k in r for k in ("salao", "barbear", "estetic", "cabelo", "unha", "spa")): return "beleza"
    if any(k in r for k in ("escola", "colegio", "creche", "infantil", "ensino", "curso")): return "escola"
    return "padrao"

def sl(t):
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
    return (re.sub(r"[^a-zA-Z0-9]+", "-", t).strip("-").lower())[:50] or "site"

def contatos(tel):
    d = re.sub(r"\D", "", tel or "")
    if d and not d.startswith("55"): d = "55" + d
    cel = len(d) == 13 and d[4] == "9"
    return ("tel:+" + d if d else ""), ("https://wa.me/" + d if cel else "")

# ─────────── achar a cor da marca ───────────

def _rgb(h, s, l):
    r, g, b = colorsys.hls_to_rgb(h % 1.0, l, s)
    return (r * 255, g * 255, b * 255)

def _hex(rgb):
    return "#%02X%02X%02X" % tuple(max(0, min(255, round(c))) for c in rgb)

def _luz_percebida(rgb):
    def canal(c):
        c = c / 255
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    r, g, b = (canal(c) for c in rgb)
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

def _contraste(a, b):
    la, lb = _luz_percebida(a), _luz_percebida(b)
    return (max(la, lb) + 0.05) / (min(la, lb) + 0.05)

def paleta_das_fotos(paths, reserva):
    """Procura nas primeiras fotos a cor que a loja usa de verdade: toldo, placa,
    fachada pintada, logo. Ignora ceu, asfalto, parede branca e sombra, que
    aparecem em toda foto e nao dizem nada sobre a marca.

    Devolve (hex, origem). Origem e 'foto' quando achou cor propria e 'nicho'
    quando caiu na cor de reserva."""
    faixas, aproveitados, olhados = {}, 0, 0
    for peso, p in zip((3, 2, 1, 1), paths[:4]):
        try:
            im = Image.open(p).convert("RGB")
        except Exception:
            continue
        im.thumbnail((150, 150))
        larg, alt = im.size
        im = im.crop((0, int(alt * 0.15), larg, alt))   # a faixa de cima quase sempre e ceu
        for r, g, b in im.getdata():
            olhados += 1
            h, l, s = colorsys.rgb_to_hls(r / 255, g / 255, b / 255)
            if s < 0.30 or not (0.20 <= l <= 0.80):
                continue                                # cinza, branco estourado ou sombra
            aproveitados += 1
            faixa = int(h * 30)                         # 30 fatias de 12 graus
            acc = faixas.setdefault(faixa, [0.0, 0.0, 0])
            acc[0] += peso * s                          # a primeira foto pesa mais
            acc[1] += s
            acc[2] += 1
    if not faixas or not olhados or aproveitados < olhados * 0.04:
        return reserva, "nicho"
    faixa = max(faixas, key=lambda f: faixas[f][0])
    _, soma_sat, quantos = faixas[faixa]
    h = (faixa + 0.5) / 30
    return _hex(_rgb(h, min(0.70, max(0.34, soma_sat / quantos)), 0.45)), "foto"

def derivar(destaque):
    """Todas as cores do site saem do mesmo matiz — por isso combinam entre si.
    As luzes sao fixas e o destaque e escurecido ate passar em 4.5:1 com texto
    branco, entao nenhum site sai com botao ilegivel."""
    r, g, b = (int(destaque[i:i+2], 16) / 255 for i in (1, 3, 5))
    h, _, s = colorsys.rgb_to_hls(r, g, b)
    vivo = min(0.66, max(0.36, s))
    luz = 0.44
    forte = _rgb(h, vivo, luz)
    while _contraste(forte, (255, 255, 255)) < 4.5 and luz > 0.16:
        luz -= 0.02
        forte = _rgb(h, vivo, luz)
    tinta = _rgb(h, 0.22, 0.11)
    return {
        "destaque":       _hex(forte),                   # botoes e detalhes
        "destaque_claro": _hex(_rgb(h, vivo, 0.70)),     # o destaque sobre a faixa escura
        "tinta":          _hex(tinta),                   # quase preto, com o matiz da marca
        "tinta_rgb":      ", ".join(str(round(c)) for c in tinta),
        "fundo":          _hex(_rgb(h, 0.14, 0.972)),    # quase branco, mesmo matiz
        "suave":          _hex(_rgb(h, 0.12, 0.915)),    # bordas e espera de imagem
    }

# ─────────── template do site ───────────
SITE = Template("""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>$nome — $cidade</title>
<meta name="description" content="$nome em $cidade. Nota $nota no Google, $aval avaliações. $endereco">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;700&family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
*,*::before,*::after{box-sizing:border-box}
body,h1,h2,h3,p,ol,li,figure,dl,dd{margin:0;padding:0}
:root{
  --tinta:$tinta; --tinta-rgb:$tinta_rgb; --fundo:$fundo;
  --destaque:$destaque; --destaque-claro:$destaque_claro; --suave:$suave;
  --carta:#fff;
  --linha:rgba(var(--tinta-rgb),.13);
  --sutil:rgba(var(--tinta-rgb),.62);
  --tec:"Space Grotesk",system-ui,sans-serif;
  --txt:"Inter",system-ui,-apple-system,sans-serif;
  --env:min(1120px,100% - 44px);
  --raio:10px;
}
html{scroll-behavior:smooth;-webkit-text-size-adjust:100%}
body{background:var(--fundo);color:var(--tinta);font-family:var(--txt);font-size:16px;
  line-height:1.6;-webkit-font-smoothing:antialiased}
.env{width:var(--env);margin-inline:auto}
img{display:block;max-width:100%}
a{color:inherit}
:focus-visible{outline:2px solid var(--destaque);outline-offset:3px;border-radius:4px}
.rotulo{font-size:.72rem;font-weight:600;letter-spacing:.12em;
  text-transform:uppercase;color:var(--destaque)}

/* topo */
.topo{position:sticky;top:0;z-index:40;background:rgba(255,255,255,.85);
  backdrop-filter:saturate(150%) blur(14px);border-bottom:1px solid var(--linha)}
.topo .env{display:flex;align-items:center;justify-content:space-between;gap:20px;height:64px}
.marca{font-family:var(--tec);font-weight:700;font-size:1.02rem;letter-spacing:-.02em;
  white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.acoes{display:flex;align-items:center;gap:14px;flex-shrink:0}
.link-tel{font-weight:600;font-size:.9rem;text-decoration:none;white-space:nowrap}
.pilula{display:inline-flex;align-items:center;background:var(--destaque);color:#fff;
  text-decoration:none;font-weight:600;font-size:.86rem;padding:10px 18px;border-radius:100px;
  white-space:nowrap;transition:filter .18s,transform .18s}
.pilula:hover{filter:brightness(1.09);transform:translateY(-1px)}

/* abertura */
.capa{width:100%;aspect-ratio:16/7;max-height:56vh;object-fit:cover;background:var(--suave)}
.abertura{padding:46px 0 0}
h1{font-family:var(--tec);font-size:clamp(1.9rem,4.4vw,3rem);line-height:1.1;
  letter-spacing:-.03em;font-weight:700;margin-top:12px}
.onde{margin-top:13px;color:var(--sutil);font-size:1.02rem;max-width:46ch}
.selo{display:inline-flex;align-items:center;gap:8px;margin-top:22px;padding:8px 16px;
  border:1px solid var(--linha);border-radius:100px;background:var(--carta);font-size:.9rem}
.selo b{font-family:var(--tec);font-weight:700}
.estrela{color:var(--destaque)}

/* numeros */
.numeros{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:1px;
  background:var(--linha);border:1px solid var(--linha);border-radius:var(--raio);
  overflow:hidden;margin-top:46px}
.numeros>div{background:var(--carta);padding:20px 22px}
.numeros dt{font-size:.68rem;font-weight:600;letter-spacing:.1em;
  text-transform:uppercase;color:var(--sutil)}
.numeros dd{font-family:var(--tec);margin-top:6px;font-size:1.35rem;
  font-weight:700;letter-spacing:-.02em}

/* mosaico */
.mosaico{display:grid;grid-template-columns:repeat(4,1fr);gap:9px;margin-top:54px}
.mosaico figure{aspect-ratio:1;overflow:hidden;border-radius:var(--raio);background:var(--suave)}
.mosaico figure:first-child{grid-column:span 2;grid-row:span 2}
.mosaico img{width:100%;height:100%;object-fit:cover;transition:transform .6s cubic-bezier(.2,.7,.3,1)}
.mosaico figure:hover img{transform:scale(1.04)}

/* servicos */
.servicos{background:var(--tinta);color:var(--fundo);margin-top:92px;padding:84px 0}
.servicos .rotulo{color:var(--destaque-claro)}
.servicos h2{font-family:var(--tec);font-size:clamp(1.5rem,3vw,2.1rem);
  letter-spacing:-.03em;font-weight:700;margin-top:10px}
.servicos ol{list-style:none;display:grid;
  grid-template-columns:repeat(auto-fit,minmax(250px,1fr));gap:38px;margin-top:48px}
.conta{display:block;font-family:var(--tec);font-size:.78rem;font-weight:700;letter-spacing:.1em;
  color:var(--destaque-claro);padding-bottom:13px;border-bottom:1px solid rgba(255,255,255,.16)}
.servicos h3{font-family:var(--tec);font-size:1.12rem;font-weight:700;
  letter-spacing:-.02em;margin:15px 0 7px}
.servicos li p{opacity:.74;font-size:.95rem}

/* visita */
.visita{margin-top:92px;padding-bottom:104px;display:grid;
  grid-template-columns:1fr 1fr;gap:56px;align-items:start}
.visita h2{font-family:var(--tec);font-size:clamp(1.6rem,3.2vw,2.3rem);
  letter-spacing:-.03em;font-weight:700;margin-top:10px}
.dados{margin-top:26px;border-top:1px solid var(--linha)}
.dados>div{padding:17px 0;border-bottom:1px solid var(--linha)}
.dados dt{font-size:.68rem;font-weight:600;letter-spacing:.1em;
  text-transform:uppercase;color:var(--sutil)}
.dados dd{margin-top:5px;font-weight:500;font-size:1.02rem}
.botoes{display:flex;gap:11px;flex-wrap:wrap;margin-top:30px}
.btn{display:inline-flex;align-items:center;justify-content:center;padding:14px 27px;
  border-radius:100px;text-decoration:none;font-weight:600;font-size:.92rem;transition:.18s}
.btn-cheio{background:var(--destaque);color:#fff}
.btn-cheio:hover{filter:brightness(1.09);transform:translateY(-1px)}
.btn-vazio{border:1.5px solid var(--linha)}
.btn-vazio:hover{border-color:var(--tinta)}
.dupla{display:grid;grid-template-columns:1fr 1fr;gap:9px}
.dupla figure{aspect-ratio:3/4;overflow:hidden;border-radius:var(--raio);background:var(--suave)}
.dupla img{width:100%;height:100%;object-fit:cover}

/* rodape */
footer{border-top:1px solid var(--linha);padding:28px 0 32px}
footer .env{display:flex;justify-content:space-between;gap:14px;flex-wrap:wrap;
  color:var(--sutil);font-size:.86rem}

/* barra do celular */
.barra{position:fixed;left:0;right:0;bottom:0;z-index:50;display:none;gap:9px;padding:10px 12px;
  background:rgba(255,255,255,.95);backdrop-filter:blur(12px);border-top:1px solid var(--linha)}
.barra a{flex:1;text-align:center;padding:14px;border-radius:100px;
  text-decoration:none;font-weight:600;font-size:.92rem}

@media(max-width:860px){
  .visita{grid-template-columns:1fr;gap:34px}
  .mosaico{grid-template-columns:repeat(2,1fr)}
  .capa{aspect-ratio:4/3;max-height:none}
  .servicos,.visita{margin-top:70px}
  .link-tel{display:none}
  .barra{display:flex}
  body{padding-bottom:76px}
}
@media(prefers-reduced-motion:reduce){*{animation:none!important;transition:none!important}}
</style>
</head>
<body>

<header class="topo">
  <div class="env">
    <span class="marca">$nome</span>
    <div class="acoes">
      <a class="link-tel" href="$tel">$telefone</a>
      <a class="pilula" href="$acao_href">$acao_texto</a>
    </div>
  </div>
</header>

<img class="capa" src="$capa" alt="$nome">

<main>
  <section class="env abertura">
    <p class="rotulo">$rotulo</p>
    <h1>$nome</h1>
    <p class="onde">$bairro</p>
    <p class="selo"><span class="estrela">★</span> <b>$nota</b> no Google · $aval avaliações</p>

    <dl class="numeros">
      <div><dt>Nota</dt><dd>$nota</dd></div>
      <div><dt>Avaliações</dt><dd>$aval</dd></div>
      <div><dt>Categoria</dt><dd>$categoria</dd></div>
    </dl>
  </section>

  $bloco_mosaico

  <section class="servicos">
    <div class="env">
      <p class="rotulo">O que você encontra aqui</p>
      <h2>Do jeito que a gente trabalha</h2>
      <ol>$cards</ol>
    </div>
  </section>

  <section class="env visita">
    <div>
      <p class="rotulo">Onde estamos</p>
      <h2>Venha até a gente</h2>
      <dl class="dados">
        <div><dt>Endereço</dt><dd>$endereco</dd></div>
        <div><dt>Telefone</dt><dd>$telefone</dd></div>
        $bloco_insta
      </dl>
      <div class="botoes">
        <a class="btn btn-cheio" href="$acao_href">$acao_texto</a>
        <a class="btn btn-vazio" href="$maps" target="_blank" rel="noopener">Ver no mapa</a>
      </div>
    </div>
    $bloco_dupla
  </section>
</main>

<footer>
  <div class="env"><span>$nome</span><span>$cidade</span></div>
</footer>

<nav class="barra">
  <a href="$tel" style="background:var(--suave);color:var(--tinta)">Ligar</a>
  <a href="$acao_href" style="background:var(--destaque);color:#fff">$acao_texto</a>
</nav>

</body>
</html>""")

# ─────────── gerar tudo ───────────
raiz = base / "sites"
if raiz.exists(): shutil.rmtree(raiz)
raiz.mkdir(parents=True)

acervo = []
for l in leads:
    if not l["pasta_fotos"]: continue
    fam = familia(l["ramo_busca"] + " " + l["categoria_google"])
    slug = sl(l["nome"])
    pasta = raiz / slug
    (pasta / "fotos").mkdir(parents=True, exist_ok=True)

    fotos = sorted(Path(l["pasta_fotos"]).glob("*.jpg"))
    for f in fotos: shutil.copy(f, pasta / "fotos" / f.name)
    if not fotos: continue
    nomes = [f"fotos/{f.name}" for f in fotos]

    destaque, origem = paleta_das_fotos([pasta / "fotos" / f.name for f in fotos], NICHO[fam])
    cor = derivar(destaque)

    nome_seguro = esc(l["nome"], quote=True)

    def figura(caminho):
        return f'<figure><img src="{caminho}" alt="{nome_seguro}" loading="lazy"></figure>'

    bloco_mosaico = ""
    if len(nomes) > 2:
        bloco_mosaico = ('<section class="env"><div class="mosaico">'
                         + "".join(figura(n) for n in nomes[1:6]) + "</div></section>")

    par = nomes[6:8] or nomes[1:3]
    bloco_dupla = f'<div class="dupla">{"".join(figura(n) for n in par)}</div>' if par else "<div></div>"

    cards = "".join(
        f'<li><span class="conta">{i:02d}</span><h3>{esc(t)}</h3><p>{esc(d)}</p></li>'
        for i, (t, d) in enumerate(SERVICOS[fam], 1)
    )

    tel_href, zap = contatos(l["telefone"])
    nota_br = str(l["nota"]).replace(".", ",")
    partes = [p.strip() for p in l["endereco"].split(",")]

    html = SITE.substitute(
        nome=nome_seguro, cidade=esc(CONFIG["cidade"]),
        nota=nota_br, aval=l["avaliacoes"],
        categoria=esc(l["categoria_google"] or l["ramo_busca"] or "Comércio local"),
        rotulo=esc((l["categoria_google"] or l["ramo_busca"]).upper()),
        endereco=esc(l["endereco"]),
        bairro=esc(", ".join(partes[1:3]) if len(partes) > 2 else l["endereco"]),
        telefone=esc(l["telefone"] or "-"), tel=tel_href or "#",
        maps=l["maps"] or "#", capa=nomes[0],
        cards=cards, bloco_mosaico=bloco_mosaico, bloco_dupla=bloco_dupla,
        bloco_insta=f'<div><dt>Instagram</dt><dd>{esc(l["instagram"])}</dd></div>' if l["instagram"] else "",
        acao_href=zap or tel_href or "#",
        acao_texto="Chamar no WhatsApp" if zap else "Ligar agora",
        **cor,
    )

    (pasta / "index.html").write_text(html, encoding="utf-8")

    tel_num = re.sub(r"\D", "", l["telefone"] or "")
    if tel_num and not tel_num.startswith("55"): tel_num = "55" + tel_num
    acervo.append({
        "slug": slug, "nome": l["nome"], "id": l["place_id"],
        "ramo": l["categoria_google"] or l["ramo_busca"],
        "nota": l["nota"], "aval": l["avaliacoes"], "km": l["km_do_centro"],
        "tel": l["telefone"] or "", "telnum": tel_num,
        "zap": tel_num if (len(tel_num) == 13 and tel_num[4] == "9") else "",
        "insta": l["instagram"], "end": l["endereco"], "maps": l["maps"],
        "capa": f"sites/{slug}/{nomes[0]}", "fotos": len(fotos),
        "cor": cor["destaque"], "origem": origem,
        "html": html.replace('src="fotos/', f'src="sites/{slug}/fotos/'),
    })
    print(f"  {l['nome']:<34} {cor['destaque']}  ({origem})")

print(f"\n{len(acervo)} sites gerados.")
print(f"{sum(1 for a in acervo if a['origem'] == 'foto')} com cor tirada das fotos, "
      f"{sum(1 for a in acervo if a['origem'] != 'foto')} com cor do nicho.\n")

In [ ]:
#@title 📋 Painel de prospecção (rode depois de gerar os sites)

import json, re, unicodedata
from pathlib import Path

def sl(t):
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
    return (re.sub(r"[^a-zA-Z0-9]+", "-", t).strip("-").lower())[:50] or "site"

por_id = {a["id"]: a for a in acervo}

dados = []
for l in leads:
    a = por_id.get(l["place_id"])
    thumb = ""
    if l["pasta_fotos"]:
        p = Path(l["pasta_fotos"])
        fotos = sorted(p.glob("*.jpg"))
        if fotos:
            thumb = str(Path(l["pasta_fotos"]).relative_to(base) / fotos[0].name).replace("\\", "/")
    site = f"sites/{a['slug']}/index.html" if a else ""
    tel = re.sub(r"\D", "", l["telefone"] or "")
    if tel and not tel.startswith("55"):
        tel = "55" + tel
    dados.append({
        "nome": l["nome"], "tel": l["telefone"] or "", "telnum": tel,
        "zap": tel if (len(tel) == 13 and tel[4] == "9") else "",
        "nota": l["nota"], "aval": l["avaliacoes"], "km": l["km_do_centro"],
        "insta": l["instagram"], "end": l["endereco"], "maps": l["maps"],
        "ramo": l["categoria_google"] or l["ramo_busca"],
        "fotos": l["qtd_fotos"], "thumb": thumb, "site": site,
        "id": l["place_id"],
        "cor": a["cor"] if a else "",
        "origem": a["origem"] if a else "",
    })

HTML = """<!DOCTYPE html>
<html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Prospeccao — __CIDADE__</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;700&family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
<style>
:root{
 --fundo:#ECEDE8; --carta:#fff; --tinta:#161A18; --sutil:#6E756F;
 --linha:#D7DAD3; --feito:#185C4A; --anda:#A8760A; --frio:#9AA09B;
 --tec:"Space Grotesk",system-ui,sans-serif; --txt:"Inter",system-ui,sans-serif;
}
*{box-sizing:border-box;margin:0;padding:0}
body{background:var(--fundo);color:var(--tinta);font-family:var(--txt);line-height:1.5}
.env{max-width:1280px;margin:0 auto;padding:0 22px}

header{border-bottom:1px solid var(--linha);padding:26px 0 0;position:sticky;top:0;background:var(--fundo);z-index:10}
.titulo{display:flex;justify-content:space-between;align-items:baseline;gap:18px;flex-wrap:wrap}
h1{font-family:var(--tec);font-size:1.45rem;font-weight:700;letter-spacing:-.02em}
.contexto{font-size:.84rem;color:var(--sutil)}

.trilho{display:flex;height:6px;border-radius:3px;overflow:hidden;margin:18px 0 10px;background:var(--linha)}
.trilho div{transition:width .4s cubic-bezier(.2,.7,.3,1)}
.legenda{display:flex;gap:20px;font-size:.78rem;color:var(--sutil);padding-bottom:16px;flex-wrap:wrap}
.legenda b{font-family:var(--tec);color:var(--tinta)}
.ponto{display:inline-block;width:8px;height:8px;border-radius:50%;margin-right:6px}

.controles{display:flex;gap:9px;padding:14px 0 20px;flex-wrap:wrap;align-items:center}
input[type=search],select{font-family:var(--txt);font-size:.88rem;padding:10px 14px;border:1px solid var(--linha);
 background:var(--carta);border-radius:7px;color:var(--tinta)}
input[type=search]{flex:1;min-width:190px}
input:focus-visible,select:focus-visible,button:focus-visible,a:focus-visible{outline:2px solid var(--feito);outline-offset:2px}
.chip{font-family:var(--txt);font-size:.8rem;font-weight:600;padding:9px 15px;border:1px solid var(--linha);
 background:var(--carta);border-radius:100px;cursor:pointer;transition:.15s}
.chip[aria-pressed=true]{background:var(--tinta);color:var(--fundo);border-color:var(--tinta)}

.grade{display:grid;grid-template-columns:repeat(auto-fill,minmax(310px,1fr));gap:14px;padding-bottom:70px}
.carta{background:var(--carta);border:1px solid var(--linha);border-left:4px solid var(--frio);
 border-radius:9px;overflow:hidden;display:flex;flex-direction:column;transition:.18s}
.carta:hover{transform:translateY(-2px);box-shadow:0 8px 24px rgba(0,0,0,.07)}
.carta[data-s=anda]{border-left-color:var(--anda)}
.carta[data-s=feito]{border-left-color:var(--feito)}
.foto{aspect-ratio:16/9;background:var(--linha);position:relative}
.foto img{width:100%;height:100%;object-fit:cover;display:block}
.qtd{position:absolute;right:9px;bottom:9px;background:rgba(0,0,0,.72);color:#fff;font-family:var(--tec);
 font-size:.68rem;font-weight:500;padding:3px 8px;border-radius:4px}
.corpo{padding:15px 16px 16px;display:flex;flex-direction:column;gap:11px;flex:1}
.nome{font-family:var(--tec);font-size:1.02rem;font-weight:700;letter-spacing:-.015em;line-height:1.25}
.ramo{font-size:.72rem;color:var(--sutil);text-transform:uppercase;letter-spacing:.07em;margin-top:3px}
.metricas{display:flex;gap:16px;align-items:baseline;padding:9px 0;border-top:1px solid var(--linha);border-bottom:1px solid var(--linha)}
.met{font-family:var(--tec);font-size:1.12rem;font-weight:700}
.met small{display:block;font-family:var(--txt);font-size:.66rem;font-weight:500;color:var(--sutil);
 text-transform:uppercase;letter-spacing:.07em}
.cor{display:flex;align-items:center;gap:6px;font-size:.71rem;color:var(--sutil);margin-top:5px}
.cor i{width:11px;height:11px;border-radius:3px;flex-shrink:0;border:1px solid rgba(0,0,0,.12)}
.dados{font-size:.83rem;color:var(--sutil);display:flex;flex-direction:column;gap:3px}
.dados b{color:var(--tinta);font-weight:600}
.acoes{display:flex;gap:6px;flex-wrap:wrap;margin-top:auto}
.bt{font-family:var(--txt);font-size:.79rem;font-weight:600;padding:8px 13px;border-radius:6px;
 text-decoration:none;border:1px solid var(--linha);background:var(--carta);color:var(--tinta);cursor:pointer;transition:.15s}
.bt:hover{background:var(--fundo)}
.bt.forte{background:var(--tinta);color:var(--fundo);border-color:var(--tinta)}
.status{display:flex;gap:5px;margin-top:9px}
.status button{flex:1;font-size:.72rem;font-weight:600;padding:7px;border-radius:5px;border:1px solid var(--linha);
 background:var(--carta);cursor:pointer;color:var(--sutil)}
.status button[aria-pressed=true]{background:var(--tinta);color:var(--fundo);border-color:var(--tinta)}
textarea{width:100%;font-family:var(--txt);font-size:.82rem;padding:9px;border:1px solid var(--linha);
 border-radius:6px;resize:vertical;min-height:38px;background:var(--fundo);color:var(--tinta)}
.vazio{text-align:center;padding:70px 20px;color:var(--sutil)}
@media(prefers-reduced-motion:reduce){*{transition:none!important}}
</style></head>
<body>
<header><div class="env">
 <div class="titulo">
   <h1>Prospeccao __CIDADE__</h1>
   <span class="contexto">__TOTAL__ comercios sem site · raio de __RAIO__ km</span>
 </div>
 <div class="trilho" id="trilho"></div>
 <div class="legenda" id="legenda"></div>
 <div class="controles">
   <input type="search" id="busca" placeholder="Buscar por nome, bairro ou rua">
   <select id="ordem">
     <option value="aval">Mais avaliacoes</option>
     <option value="nota">Melhor nota</option>
     <option value="km">Mais perto</option>
     <option value="nome">Nome A-Z</option>
   </select>
   <button class="chip" id="fInsta" aria-pressed="false">Com Instagram</button>
   <button class="chip" id="fAbertos" aria-pressed="false">So a fazer</button>
   <button class="bt" id="exportar">Exportar CSV</button>
 </div>
</div></header>

<main class="env"><div class="grade" id="grade"></div></main>

<script>
const LEADS = __DADOS__;
const chave = "prospector:" + "__SLUG__";
let estado = JSON.parse(localStorage.getItem(chave) || "{}");
const salvar = () => localStorage.setItem(chave, JSON.stringify(estado));
const est = id => estado[id] || {s:"frio", nota:""};

const rotulos = {frio:["A fazer","var(--frio)"], anda:["Contatado","var(--anda)"], feito:["Fechado","var(--feito)"]};

function barra(){
  const c = {frio:0, anda:0, feito:0};
  LEADS.forEach(l => c[est(l.id).s]++);
  const t = LEADS.length || 1;
  document.getElementById("trilho").innerHTML =
    ["feito","anda","frio"].map(k => `<div style="width:${c[k]/t*100}%;background:${rotulos[k][1]}"></div>`).join("");
  document.getElementById("legenda").innerHTML =
    ["frio","anda","feito"].map(k =>
      `<span><i class="ponto" style="background:${rotulos[k][1]}"></i>${rotulos[k][0]} <b>${c[k]}</b></span>`).join("");
}

function desenhar(){
  const q = document.getElementById("busca").value.toLowerCase().trim();
  const ordem = document.getElementById("ordem").value;
  const soInsta = document.getElementById("fInsta").getAttribute("aria-pressed") === "true";
  const soAbertos = document.getElementById("fAbertos").getAttribute("aria-pressed") === "true";

  let lista = LEADS.filter(l => {
    if (q && !(l.nome + " " + l.end).toLowerCase().includes(q)) return false;
    if (soInsta && !l.insta) return false;
    if (soAbertos && est(l.id).s !== "frio") return false;
    return true;
  });

  lista.sort((a,b) => ordem === "nota" ? b.nota - a.nota
    : ordem === "km" ? a.km - b.km
    : ordem === "nome" ? a.nome.localeCompare(b.nome)
    : b.aval - a.aval);

  const g = document.getElementById("grade");
  if (!lista.length){ g.innerHTML = '<p class="vazio">Nenhum lead com esses filtros.</p>'; barra(); return; }

  g.innerHTML = lista.map(l => {
    const e = est(l.id);
    const foto = l.thumb
      ? `<div class="foto"><img src="${l.thumb}" alt="" loading="lazy"><span class="qtd">${l.fotos} fotos</span></div>`
      : `<div class="foto"></div>`;
    const zap = l.zap ? `<a class="bt" href="https://wa.me/${l.zap}" target="_blank" rel="noopener">WhatsApp</a>` : "";
    const insta = l.insta ? `<div>Instagram <b>${l.insta}</b></div>` : "";
    return `<article class="carta" data-s="${e.s}">
      ${foto}
      <div class="corpo">
        <div>
          <div class="nome">${l.nome}</div>
          <div class="ramo">${l.ramo}</div>
          ${l.cor ? `<div class="cor"><i style="background:${l.cor}"></i>cor ${l.origem === "foto" ? "tirada da foto" : "padrao do nicho"}</div>` : ""}
        </div>
        <div class="metricas">
          <div class="met">${l.nota.toFixed(1).replace(".",",")}<small>Nota</small></div>
          <div class="met">${l.aval}<small>Avaliacoes</small></div>
          <div class="met">${String(l.km).replace(".",",")}<small>km</small></div>
        </div>
        <div class="dados">
          <div><b>${l.tel || "sem telefone"}</b></div>
          ${insta}
          <div>${l.end}</div>
        </div>
        <div class="acoes">
          ${l.site ? `<a class="bt forte" href="${l.site}" target="_blank" rel="noopener">Ver o site</a>`
                   : `<span class="bt" style="opacity:.45">Sem fotos</span>`}
          ${l.telnum ? `<a class="bt" href="tel:+${l.telnum}">Ligar</a>` : ""}
          ${zap}
          <a class="bt" href="${l.maps}" target="_blank" rel="noopener">Mapa</a>
        </div>
        <div class="status" data-id="${l.id}">
          ${["frio","anda","feito"].map(k =>
            `<button data-k="${k}" aria-pressed="${e.s===k}">${rotulos[k][0]}</button>`).join("")}
        </div>
        <textarea data-nota="${l.id}" placeholder="Anotacoes da visita">${e.nota||""}</textarea>
      </div>
    </article>`;
  }).join("");
  barra();
}

document.addEventListener("click", ev => {
  const b = ev.target.closest(".status button");
  if (b){
    const id = b.parentElement.dataset.id;
    estado[id] = {...est(id), s: b.dataset.k};
    salvar(); desenhar(); return;
  }
  const c = ev.target.closest(".chip");
  if (c){ c.setAttribute("aria-pressed", c.getAttribute("aria-pressed") !== "true"); desenhar(); }
});
document.addEventListener("input", ev => {
  if (ev.target.dataset.nota){
    const id = ev.target.dataset.nota;
    estado[id] = {...est(id), nota: ev.target.value};
    salvar();
  }
});
document.getElementById("busca").addEventListener("input", desenhar);
document.getElementById("ordem").addEventListener("change", desenhar);
document.getElementById("exportar").addEventListener("click", () => {
  const linhas = [["Nome","Telefone","Endereco","Nota","Avaliacoes","Km","Instagram","Status","Anotacoes"]];
  LEADS.forEach(l => {
    const e = est(l.id);
    linhas.push([l.nome,l.tel,l.end,l.nota.toFixed(1).replace(".",","),l.aval,
      String(l.km).replace(".",","),l.insta,rotulos[e.s][0],(e.nota||"").replace(/\\n/g," ")]);
  });
  const csv = "\\uFEFF" + linhas.map(r => r.map(c => `"${String(c).replace(/"/g,'""')}"`).join(";")).join("\\n");
  const a = document.createElement("a");
  a.href = URL.createObjectURL(new Blob([csv], {type:"text/csv"}));
  a.download = "prospeccao.csv"; a.click();
});

desenhar();
</script>
</body></html>"""

html = (HTML
        .replace("__DADOS__", json.dumps(dados, ensure_ascii=False))
        .replace("__CIDADE__", CONFIG["cidade"])
        .replace("__TOTAL__", str(len(dados)))
        .replace("__RAIO__", str(int(CONFIG["raio_km"])))
        .replace("__SLUG__", sl(CONFIG["cidade"])))

(base / "painel.html").write_text(html, encoding="utf-8")

print(f"Painel gerado com {len(dados)} leads.")
print(f"{sum(1 for d in dados if d['site'])} com site pronto, "
      f"{sum(1 for d in dados if not d['site'])} sem fotos para montar site.")
print("\nO .zip com tudo sai na última célula.")

In [ ]:
#@title 🤖 Editar os sites conversando com o Claude

# A chave do Claude tambem fica no cofre do Colab, nunca escrita aqui:
#   menu da esquerda -> 🔑 -> Adicionar novo secret
#   Nome:  ANTHROPIC_API_KEY
#   Valor: sua chave (console.anthropic.com -> API keys)
#
# O Claude roda AQUI no Colab, nao dentro do site. Assim a chave nunca
# entra no .zip que voce manda para o cliente.

!pip install -q anthropic

import re, shutil
from pathlib import Path
import anthropic

MODELO = "claude-opus-5"

try:
    from google.colab import userdata
    CHAVE_CLAUDE = (userdata.get("ANTHROPIC_API_KEY") or "").strip()
except Exception:
    CHAVE_CLAUDE = ""

cliente = anthropic.Anthropic(api_key=CHAVE_CLAUDE) if CHAVE_CLAUDE else None

INSTRUCOES = """Você edita páginas HTML de sites de comércios de bairro brasileiros.

Devolva SEMPRE o arquivo HTML completo, do <!DOCTYPE html> até </html>.
Nada antes, nada depois, sem cercas de código, sem explicação.

Regras que não se quebram:
- Mantenha os caminhos das imagens (src="fotos/...") exatamente como estão.
- Mantenha os links de telefone (tel:), WhatsApp (wa.me) e mapa funcionando.
- Mantenha as variáveis de cor (--tinta, --fundo, --destaque, --suave, --tinta-rgb)
  exatamente como estão, a menos que o pedido seja justamente sobre cor.
- Não invente serviço, preço, horário, promoção nem promessa que não esteja
  no HTML atual. Se o pedido exigir um dado que você não tem, deixe o texto
  genérico em vez de inventar.
- Português do Brasil, com acentos.
- O site tem que continuar funcionando no celular."""


def _so_o_html(texto):
    """Tira cerca de codigo e qualquer conversa em volta."""
    t = texto.strip()
    t = re.sub(r"^```(?:html)?\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    i, j = t.find("<!DOCTYPE"), t.rfind("</html>")
    if i == -1:
        i = t.find("<html")
    return t[i:j + 7] if (i != -1 and j != -1) else ""


def editar(slug, pedido):
    """Pede uma mudanca no site de um lead. Guarda o original na primeira vez,
    entao da sempre para voltar atras com desfazer(slug)."""
    if cliente is None:
        print("⚠️  Falta a chave do Claude no cofre do Colab (ANTHROPIC_API_KEY).")
        return

    pasta = raiz / slug
    alvo = pasta / "index.html"
    if not alvo.exists():
        print(f"Não achei o site '{slug}'.")
        print("Os que existem:", ", ".join(sorted(p.name for p in raiz.iterdir() if p.is_dir())))
        return

    original = pasta / "index.original.html"
    if not original.exists():
        shutil.copy(alvo, original)          # guarda o primeiro, uma vez so

    antes = alvo.read_text(encoding="utf-8")
    print(f"Pedindo ao Claude: {pedido}\n")

    try:
        with cliente.messages.stream(
            model=MODELO,
            max_tokens=32000,
            system=INSTRUCOES,
            messages=[{
                "role": "user",
                "content": f"Mudança pedida: {pedido}\n\nHTML atual do site:\n\n{antes}",
            }],
        ) as fluxo:
            resposta = fluxo.get_final_message()
    except anthropic.AuthenticationError:
        print("⚠️  A chave do Claude não foi aceita. Confira o secret ANTHROPIC_API_KEY.")
        return
    except anthropic.RateLimitError:
        print("⚠️  Muitos pedidos seguidos. Espere um minuto e tente de novo.")
        return
    except anthropic.APIConnectionError:
        print("⚠️  Sem conexão com a API. Tente de novo.")
        return
    except anthropic.APIStatusError as e:
        print(f"⚠️  A API respondeu com erro {e.status_code}.")
        return

    if resposta.stop_reason == "refusal":
        print("⚠️  O Claude preferiu não fazer essa mudança.")
        return
    if resposta.stop_reason == "max_tokens":
        print("⚠️  A resposta foi cortada no meio. O site NÃO foi alterado.")
        print("   Peça uma mudança menor de cada vez.")
        return

    texto = "".join(b.text for b in resposta.content if b.type == "text")
    novo = _so_o_html(texto)

    if not novo or len(novo) < len(antes) * 0.4:
        print("⚠️  A resposta não veio como uma página inteira. O site NÃO foi alterado.")
        return

    alvo.write_text(novo, encoding="utf-8")

    for a in acervo:
        if a["slug"] == slug:
            a["html"] = novo.replace('src="fotos/', f'src="sites/{slug}/fotos/')

    g = resposta.usage
    print(f"✓ {slug} atualizado ({len(antes):,} → {len(novo):,} caracteres)")
    print(f"  tokens: {g.input_tokens:,} entrada, {g.output_tokens:,} saída")
    print(f"  para voltar atrás: desfazer('{slug}')")


def desfazer(slug):
    """Volta o site para como ele saiu do gerador."""
    pasta = raiz / slug
    original = pasta / "index.original.html"
    if not original.exists():
        print(f"'{slug}' não foi editado ainda — não há o que desfazer.")
        return
    shutil.copy(original, pasta / "index.html")
    print(f"✓ {slug} voltou ao original.")


def sites():
    """Lista os apelidos que voce usa em editar()."""
    for a in acervo:
        marca = " (editado)" if (raiz / a["slug"] / "index.original.html").exists() else ""
        print(f"  {a['slug']:<50} {a['nome'][:34]}{marca}")


print("Pronto. Como usar:\n")
print("  sites()                                    → lista os apelidos")
print("  editar('nutry-pet-shop', 'seu pedido')     → muda esse site")
print("  desfazer('nutry-pet-shop')                 → volta ao original\n")
print("Exemplos de pedido:")
print("  'troque o texto dos serviços para banho, tosa e consulta veterinária'")
print("  'deixe o título mais curto e o botão do WhatsApp mais destacado'")
print("  'escreva um parágrafo de apresentação abaixo do nome'\n")

if cliente is None:
    print("⚠️  Antes de usar: ponha ANTHROPIC_API_KEY no cofre do Colab (🔑 no menu da esquerda).")
else:
    print(f"Claude conectado — modelo {MODELO}.")

In [ ]:
#@title 📥 Baixar tudo num arquivo só

import shutil
from pathlib import Path
from google.colab import files

nome = f"prospeccao-{sl(CONFIG['cidade'])}-{int(CONFIG['raio_km'])}km"
caminho = shutil.make_archive(nome, "zip", base)
tamanho = Path(caminho).stat().st_size / (1024 * 1024)

print(f"{nome}.zip — {tamanho:.1f} MB\n")
print("Dentro dele:")
print("  painel.html        abra este primeiro: é o seu painel de prospecção")
print("  leads.pdf          a lista para imprimir ou mandar por WhatsApp")
print("  leads_excel.csv    abre direto no Excel brasileiro")
print("  leads.csv          o mesmo, para importar em outros programas")
print("  sites/             um site pronto por lead, com index.html")
print("  fotos/             as fotos baixadas do Google\n")
print("Como usar: descompacte a pasta e abra o painel.html no navegador.")
print("Os links do painel só funcionam com a pasta inteira junto.\n")
print("⚠️  As fotos são de quem as enviou ao Google. Use para montar a proposta")
print("    e mostrar o preview. Só publique em site aberto depois do sim do dono.")

files.download(caminho)